# Comparação SHPM  com grupos Female e Male do IMDB




## 1. Instalação e importações



In [1]:
# Instalação opcional em ambiente novo:
%pip install -q numpy pandas pillow scipy matplotlib seaborn scikit-learn tqdm torch torchvision

!git clone https://github.com/facebookresearch/sam3.git
%pip install -q -e sam3



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
fatal: destination path 'sam3' already exists and is not an empty directory.

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install einops


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


In [3]:
!pip install decord


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


In [4]:
!pip install pycocotools


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


In [5]:
!pip install huggingface_hub


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


In [6]:
%cd sam3
%cd sam3

/workspace/sam3
/workspace/sam3/sam3


/usr/local/lib/python3.11/dist-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [7]:
import os
print(os.getcwd())

/workspace/sam3/sam3


In [8]:
from contextlib import nullcontext

In [9]:
from __future__ import annotations

import hashlib
import math
import os
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image

try:
    from IPython.display import display
except ImportError:
    def display(obj: Any) -> None:
        print(obj)

import matplotlib.pyplot as plt
from scipy.ndimage import binary_closing, binary_opening
from sklearn.preprocessing import MinMaxScaler

try:
    import seaborn as sns
except ImportError:
    sns = None

try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda iterable, **_: iterable

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)


In [10]:
from huggingface_hub import login

login()

## 2. Configuração

Todos os caminhos são relativos ao diretório de execução do notebook. O fluxo espera que o diretório atual seja sam3/sam3, como nas células iniciais com %cd.


In [11]:
REFERENCE_CSV = "imdb_shpm_reference_valid.csv"
IMAGES_FOLDER = "images_quadro/"
OUTPUT_FOLDER = "outputs_shpm_original/"
EXAMPLE_FIGURE = "Flux_SHPM.png"
SCORE_THRESHOLD = 0.5
MIN_FACE_AREA = 100
RANDOM_SEED = 42

COMPUTE_BOOTSTRAP = True
BOOTSTRAP_RESAMPLES = 1000
SIGMOID_MU = 0.88
SIGMOID_BETA = 10.0

SAM3_PROMPTS = {
    "face": "face",
    "nose": "nose",
    "mouth": "mouth",
    "l_eye": "left eye",
    "r_eye": "right eye",
    "hair": "hair",
}

parts = ["face", "nose", "mouth", "l_eye", "r_eye", "hair"]
feature_names = [
    f"{part_i}_vs_{part_j}"
    for part_i in parts
    for part_j in parts
    if part_i != part_j
]

assert len(feature_names) == 30
assert len(set(feature_names)) == 30

BAR_COLORS = {
    "Women": "#d9a7c7",
    "Men": "#93b7dc",
}

MASK_COLORS = {
    "face": (250, 184, 132),
    "nose": (141, 211, 199),
    "mouth": (255, 128, 128),
    "l_eye": (128, 177, 211),
    "r_eye": (188, 128, 189),
    "hair": (190, 174, 212),
}

OUTPUT_ROOT = Path(OUTPUT_FOLDER)
DIR_SEGMENTATIONS = OUTPUT_ROOT / "segmentations"
DIR_MASKS = OUTPUT_ROOT / "masks"
DIR_INDIVIDUAL_PLOTS = OUTPUT_ROOT / "individual_plots"
DIR_SUMMARY = OUTPUT_ROOT / "summary"
DIR_DIAGNOSTICS = OUTPUT_ROOT / "diagnostics"

RESULT_COLUMNS = [
    "filename",
    "status",
    "error_message",
    "face_area",
    "face_score",
    "nose_score",
    "mouth_score",
    "l_eye_score",
    "r_eye_score",
    "hair_score",
    "mean_cosine_female",
    "mean_cosine_male",
    "shpm_female",
    "shpm_male",
    "delta_shpm",
    "absolute_difference",
    "higher_similarity_group",
    "female_ci_lower",
    "female_ci_upper",
    "male_ci_lower",
    "male_ci_upper",
    "delta_ci_lower",
    "delta_ci_upper",
    "comparison_status",
]


def ensure_output_folders() -> None:
    for folder in [OUTPUT_ROOT, DIR_SEGMENTATIONS, DIR_MASKS, DIR_INDIVIDUAL_PLOTS, DIR_SUMMARY, DIR_DIAGNOSTICS]:
        folder.mkdir(parents=True, exist_ok=True)


def safe_stem(path: str | Path) -> str:
    stem = Path(path).stem
    stem = re.sub(r"[^A-Za-z0-9_.-]+", "_", stem).strip("._")
    return stem or "image"


def stable_seed(text: str, base_seed: int = RANDOM_SEED) -> int:
    digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
    return base_seed + int(digest[:8], 16)


np.random.seed(RANDOM_SEED)
ensure_output_folders()

print("Atributos SHPM originais:")
print(feature_names)


Atributos SHPM originais:
['face_vs_nose', 'face_vs_mouth', 'face_vs_l_eye', 'face_vs_r_eye', 'face_vs_hair', 'nose_vs_face', 'nose_vs_mouth', 'nose_vs_l_eye', 'nose_vs_r_eye', 'nose_vs_hair', 'mouth_vs_face', 'mouth_vs_nose', 'mouth_vs_l_eye', 'mouth_vs_r_eye', 'mouth_vs_hair', 'l_eye_vs_face', 'l_eye_vs_nose', 'l_eye_vs_mouth', 'l_eye_vs_r_eye', 'l_eye_vs_hair', 'r_eye_vs_face', 'r_eye_vs_nose', 'r_eye_vs_mouth', 'r_eye_vs_l_eye', 'r_eye_vs_hair', 'hair_vs_face', 'hair_vs_nose', 'hair_vs_mouth', 'hair_vs_l_eye', 'hair_vs_r_eye']


## 3. Inicialização do SAM 3

A inicialização é feita com CUDA quando disponível e CPU como fallback. Se o SAM 3 não estiver instalado, o erro é preservado e registrado para as imagens do lote, sem `continue` silencioso.


In [12]:
import inspect
from pathlib import Path

import torch
import sam3.model_builder as sam3_builder

from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor


def find_bpe_path() -> Path:
    """
    Procura o arquivo BPE relativamente ao arquivo model_builder.py,
    evitando o erro do pkg_resources.
    """

    builder_file = Path(
        inspect.getfile(sam3_builder)
    ).resolve()

    builder_folder = builder_file.parent

    filename = "bpe_simple_vocab_16e6.txt.gz"

    candidates = [
        builder_folder / "assets" / filename,
        builder_folder.parent / "assets" / filename,
        Path.cwd() / "assets" / filename,
        Path.cwd() / "sam3" / "assets" / filename,
        Path.cwd() / "sam3" / "sam3" / "assets" / filename,
    ]

    for candidate in candidates:
        candidate = candidate.resolve()

        if candidate.is_file():
            return candidate

    checked_paths = "\n".join(
        f"  - {path.resolve()}"
        for path in candidates
    )

    raise FileNotFoundError(
        "O arquivo do tokenizer do SAM 3 não foi encontrado.\n"
        "Caminhos verificados:\n"
        f"{checked_paths}"
    )


def initialize_sam3():
    device = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    bpe_path = find_bpe_path()

    print(f"Dispositivo: {device}")
    print(f"Arquivo BPE: {bpe_path}")
    print(f"Model builder: {inspect.getfile(sam3_builder)}")

    model = build_sam3_image_model(
        bpe_path=str(bpe_path),
        device=device,
    )

    model.eval()

    try:
        processor = Sam3Processor(
            model,
            confidence_threshold=SCORE_THRESHOLD,
        )
    except TypeError:
        # Compatibilidade com versões que não possuem
        # confidence_threshold no construtor.
        processor = Sam3Processor(model)

    print("SAM 3 inicializado corretamente.")

    return model, processor, device

/usr/local/lib/python3.11/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [13]:
bpe_path = find_bpe_path()

print(bpe_path)
print("Existe:", bpe_path.exists())

/workspace/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz
Existe: True


## 4. Funções de segmentação



In [14]:
def tensor_or_array_to_numpy_mask(mask: Any) -> np.ndarray:
    if torch is not None and isinstance(mask, torch.Tensor):
        arr = mask.detach().cpu().numpy()
    else:
        arr = np.asarray(mask)

    arr = np.squeeze(arr)
    if arr.ndim != 2:
        raise ValueError(f"Máscara com dimensão inesperada: {arr.shape}")

    if arr.dtype != bool:
        arr = arr > 0.5

    return arr.astype(bool)


def sam3_segment_parts(
    img_rgb: Image.Image,
    processor: Any,
    score_threshold: float = SCORE_THRESHOLD,
) -> tuple[dict[str, np.ndarray], dict[str, float], list[str]]:
    from contextlib import nullcontext

    width, height = img_rgb.size
    empty_mask = np.zeros((height, width), dtype=bool)

    part_masks: dict[str, np.ndarray] = {}
    part_scores: dict[str, float] = {}
    part_errors: list[str] = []

    if torch.cuda.is_available():
        supports_bfloat16 = getattr(
            torch.cuda,
            "is_bf16_supported",
            lambda: False,
        )()

        amp_dtype = (
            torch.bfloat16
            if supports_bfloat16
            else torch.float16
        )

        inference_context = torch.autocast(
            device_type="cuda",
            dtype=amp_dtype,
        )
    else:
        inference_context = nullcontext()

    def run_prompt(
        inference_state: Any,
        prompt: str,
    ) -> list[dict[str, Any]]:
        """Executa um prompt e devolve todas as máscaras válidas."""

        output = processor.set_text_prompt(
            state=inference_state,
            prompt=prompt,
        )

        masks = output.get("masks")
        scores = output.get("scores")

        if masks is None or scores is None or len(masks) == 0:
            return []

        scores_tensor = torch.as_tensor(scores).reshape(-1)

        if scores_tensor.numel() == 0:
            return []

        candidate_count = min(
            len(masks),
            scores_tensor.numel(),
        )

        candidates: list[dict[str, Any]] = []

        for index in range(candidate_count):
            mask = tensor_or_array_to_numpy_mask(masks[index])

            if mask.shape != (height, width):
                continue

            area = int(mask.sum())

            if area <= 0:
                continue

            y_coordinates, x_coordinates = np.where(mask)

            candidates.append({
                "mask": mask,
                "score": float(scores_tensor[index].item()),
                "area": area,
                "center_x": float(x_coordinates.mean()),
                "center_y": float(y_coordinates.mean()),
            })

        return candidates

    def mask_iou(
        mask_a: np.ndarray,
        mask_b: np.ndarray,
    ) -> float:
        intersection = np.logical_and(mask_a, mask_b).sum()
        union = np.logical_or(mask_a, mask_b).sum()

        if union == 0:
            return 0.0

        return float(intersection / union)

    def remove_duplicate_candidates(
        candidates: list[dict[str, Any]],
        maximum_iou: float = 0.50,
    ) -> list[dict[str, Any]]:
        """Remove máscaras repetidas, preservando as de maior score."""

        ordered = sorted(
            candidates,
            key=lambda candidate: candidate["score"],
            reverse=True,
        )

        unique: list[dict[str, Any]] = []

        for candidate in ordered:
            is_duplicate = any(
                mask_iou(candidate["mask"], saved["mask"])
                > maximum_iou
                for saved in unique
            )

            if not is_duplicate:
                unique.append(candidate)

        return unique

    with torch.inference_mode(), inference_context:
        inference_state = processor.set_image(img_rgb)

        # Segmenta as regiões que não são olhos.
        non_eye_parts = ["face", "nose", "mouth", "hair"]

        for part_name in non_eye_parts:
            prompt = SAM3_PROMPTS[part_name]

            try:
                candidates = run_prompt(
                    inference_state=inference_state,
                    prompt=prompt,
                )

                if not candidates:
                    part_masks[part_name] = empty_mask.copy()
                    part_scores[part_name] = np.nan
                    part_errors.append(
                        f"{part_name}: nenhuma máscara retornada "
                        f"pelo SAM 3"
                    )
                    continue

                best_candidate = max(
                    candidates,
                    key=lambda candidate: candidate["score"],
                )

                best_score = best_candidate["score"]

                part_masks[part_name] = best_candidate["mask"]
                part_scores[part_name] = best_score

                if best_score < score_threshold:
                    part_errors.append(
                        f"AVISO {part_name}: melhor score "
                        f"{best_score:.3f} abaixo do limiar "
                        f"{score_threshold:.3f}"
                    )

            except Exception as exc:
                part_masks[part_name] = empty_mask.copy()
                part_scores[part_name] = np.nan
                part_errors.append(
                    f"{part_name}: {type(exc).__name__}: {exc}"
                )

        # Segmenta os dois olhos usando um prompt genérico.
        try:
            eye_candidates: list[dict[str, Any]] = []

            for eye_prompt in ["eye", "human eye"]:
                eye_candidates.extend(
                    run_prompt(
                        inference_state=inference_state,
                        prompt=eye_prompt,
                    )
                )

            eye_candidates = remove_duplicate_candidates(
                eye_candidates
            )

            # Preserva as duas máscaras de olho com maior confiança.
            eye_candidates = sorted(
                eye_candidates,
                key=lambda candidate: candidate["score"],
                reverse=True,
            )[:2]

            if len(eye_candidates) < 2:
                part_masks["l_eye"] = empty_mask.copy()
                part_masks["r_eye"] = empty_mask.copy()
                part_scores["l_eye"] = np.nan
                part_scores["r_eye"] = np.nan

                part_errors.append(
                    "eyes: não foi possível encontrar duas "
                    "máscaras distintas de olhos"
                )

            else:
                # Ordenação da esquerda para a direita na imagem.
                eye_candidates = sorted(
                    eye_candidates,
                    key=lambda candidate: candidate["center_x"],
                )

                image_left_eye = eye_candidates[0]
                image_right_eye = eye_candidates[1]

                # Em uma face frontal, o olho direito da pessoa aparece
                # no lado esquerdo da imagem.
                part_masks["r_eye"] = image_left_eye["mask"]
                part_scores["r_eye"] = image_left_eye["score"]

                part_masks["l_eye"] = image_right_eye["mask"]
                part_scores["l_eye"] = image_right_eye["score"]

                for part_name in ["l_eye", "r_eye"]:
                    score = part_scores[part_name]

                    if score < score_threshold:
                        part_errors.append(
                            f"AVISO {part_name}: melhor score "
                            f"{score:.3f} abaixo do limiar "
                            f"{score_threshold:.3f}"
                        )

        except Exception as exc:
            part_masks["l_eye"] = empty_mask.copy()
            part_masks["r_eye"] = empty_mask.copy()
            part_scores["l_eye"] = np.nan
            part_scores["r_eye"] = np.nan

            part_errors.append(
                f"eyes: {type(exc).__name__}: {exc}"
            )

    # Garante que todas as seis regiões estejam presentes.
    for part_name in parts:
        part_masks.setdefault(
            part_name,
            empty_mask.copy(),
        )
        part_scores.setdefault(
            part_name,
            np.nan,
        )

    return part_masks, part_scores, part_errors

## 5. Limpeza e validação das máscaras



In [15]:
def clean_mask(mask: np.ndarray) -> np.ndarray:
    mask = np.asarray(mask, dtype=bool)
    opened = binary_opening(mask, structure=np.ones((3, 3), dtype=bool))
    closed = binary_closing(opened, structure=np.ones((3, 3), dtype=bool))
    return closed.astype(bool)


def clean_masks(raw_masks: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    return {part: clean_mask(raw_masks.get(part, np.zeros((1, 1), dtype=bool))) for part in parts}


def summarize_mask_areas(masks: dict[str, np.ndarray]) -> dict[str, int]:
    return {part: int(np.asarray(masks[part], dtype=bool).sum()) for part in parts}

def validate_masks(
    areas: dict[str, int],
    part_errors: list[str],
    min_face_area: int = MIN_FACE_AREA,
) -> None:
    fatal_errors = [
        error
        for error in part_errors
        if not error.startswith("AVISO ")
    ]

    if fatal_errors:
        raise ValueError("; ".join(fatal_errors))

    face_area = areas.get("face", 0)

    if face_area < min_face_area:
        raise ValueError(
            f"face inválida: área {face_area} menor que "
            f"MIN_FACE_AREA={min_face_area}"
        )

    missing_parts = [
        part
        for part in parts
        if areas.get(part, 0) <= 0
    ]

    if missing_parts:
        raise ValueError(
            "regiões essenciais ausentes após limpeza: "
            f"{missing_parts}"
        )


def save_binary_masks(image_stem: str, masks: dict[str, np.ndarray]) -> None:
    mask_dir = DIR_MASKS / image_stem
    mask_dir.mkdir(parents=True, exist_ok=True)
    for part, mask in masks.items():
        mask_img = Image.fromarray((np.asarray(mask, dtype=np.uint8) * 255), mode="L")
        mask_img.save(mask_dir / f"{part}.png")


def save_combined_overlay(image_stem: str, img_rgb: Image.Image, masks: dict[str, np.ndarray], alpha: float = 0.42) -> None:
    image_arr = np.asarray(img_rgb.convert("RGB"), dtype=float)
    overlay = image_arr.copy()

    for part, mask in masks.items():
        color = np.asarray(MASK_COLORS[part], dtype=float)
        mask_bool = np.asarray(mask, dtype=bool)
        overlay[mask_bool] = (1.0 - alpha) * overlay[mask_bool] + alpha * color

    out = np.clip(overlay, 0, 255).astype(np.uint8)
    Image.fromarray(out).save(DIR_SEGMENTATIONS / f"{image_stem}_overlay.png")


def save_segmentation_panel(
    image_stem: str,
    img_rgb: Image.Image,
    masks: dict[str, np.ndarray],
    areas: dict[str, int],
    scores: dict[str, float],
) -> None:
    fig, axes = plt.subplots(2, 3, figsize=(12, 8), dpi=160)
    axes = axes.ravel()
    base = np.asarray(img_rgb.convert("RGB"))

    for ax, part in zip(axes, parts):
        ax.imshow(base)
        color = np.asarray(MASK_COLORS[part], dtype=float) / 255.0
        colored = np.zeros((*masks[part].shape, 4), dtype=float)
        colored[..., :3] = color
        colored[..., 3] = np.asarray(masks[part], dtype=bool) * 0.55
        ax.imshow(colored)
        score = scores.get(part, np.nan)
        score_text = "nan" if not np.isfinite(score) else f"{score:.3f}"
        ax.set_title(f"{part}\nárea={areas.get(part, 0)} | score={score_text}", fontsize=10)
        ax.axis("off")

    fig.tight_layout()
    fig.savefig(DIR_SEGMENTATIONS / f"{image_stem}_panel.png", dpi=220)
    plt.close(fig)


def save_segmentation_outputs(
    image_path: Path,
    img_rgb: Image.Image,
    masks: dict[str, np.ndarray],
    areas: dict[str, int],
    scores: dict[str, float],
) -> None:
    image_stem = safe_stem(image_path)
    img_rgb.save(DIR_SEGMENTATIONS / f"{image_stem}_original.png")
    save_binary_masks(image_stem, masks)
    save_combined_overlay(image_stem, img_rgb, masks)
    save_segmentation_panel(image_stem, img_rgb, masks, areas, scores)


## 6. Extração do vetor SHPM 



In [16]:
def extract_shpm_features_from_areas(areas: dict[str, int]) -> dict[str, float]:
    values: dict[str, float] = {}

    for part_i in parts:
        for part_j in parts:
            if part_i == part_j:
                continue
            area_i = float(areas.get(part_i, 0))
            area_j = float(areas.get(part_j, 0))
            values[f"{part_i}_vs_{part_j}"] = area_i / area_j if area_j > 0 else 0.0

    assert list(values.keys()) == feature_names
    assert len(values) == 30
    return values


def make_empty_feature_row(filename: str, status: str, error_message: str = "") -> dict[str, Any]:
    row = {"filename": filename, "status": status, "error_message": error_message}
    row.update({feature: np.nan for feature in feature_names})
    return row


## 7. Carregamento e harmonização do CSV do IMDB



In [17]:
REFERENCE_COLUMN_MAP = {
    "Nose_vs_Face": "nose_vs_face",
    "Mouth_vs_Face": "mouth_vs_face",
    "Nose_vs_Mouth": "nose_vs_mouth",
}

AGGREGATED_EYE_COLUMNS = ["Eyes_vs_Face", "Eyes_vs_Nose", "Eyes_vs_Mouth"]


def safe_inverse(values: Any, eps: float = 1e-12) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    out = np.zeros_like(arr, dtype=float)
    mask = np.isfinite(arr) & (np.abs(arr) > eps)
    out[mask] = 1.0 / arr[mask]
    return out


def normalize_gender(series: pd.Series) -> pd.Series:
    cleaned = series.astype(str).str.strip().str.casefold()
    mapping = {
        "male": "Male",
        "m": "Male",
        "man": "Male",
        "men": "Male",
        "female": "Female",
        "f": "Female",
        "woman": "Female",
        "women": "Female",
    }
    return cleaned.map(mapping)


def build_reference_feature_matrix(raw_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    renamed = raw_df.rename(columns=lambda col: str(col).strip()).rename(columns=REFERENCE_COLUMN_MAP)

    features = pd.DataFrame(index=renamed.index)
    for feature in feature_names:
        if feature in renamed.columns:
            features[feature] = pd.to_numeric(renamed[feature], errors="coerce")
        else:
            features[feature] = np.nan

    reciprocal_rules = {
        "l_eye_vs_face": "face_vs_l_eye",
        "r_eye_vs_face": "face_vs_r_eye",
        "l_eye_vs_r_eye": "r_eye_vs_l_eye",
    }
    for target, source in reciprocal_rules.items():
        if source not in features.columns:
            raise ValueError(f"Não foi possível reconstruir {target}: coluna fonte ausente ({source}).")
        if features[target].isna().all():
            features[target] = safe_inverse(features[source])

    all_nan_columns = [col for col in feature_names if features[col].isna().all()]
    if all_nan_columns:
        raise ValueError(f"Colunas SHPM sem dados após harmonização: {all_nan_columns}")

    metadata = pd.DataFrame(index=renamed.index)
    metadata["filename"] = renamed["filename"].astype(str) if "filename" in renamed.columns else renamed.index.astype(str)
    metadata["gender"] = normalize_gender(renamed["gender"]) if "gender" in renamed.columns else np.nan
    metadata["area_face"] = pd.to_numeric(renamed["area_face"], errors="coerce") if "area_face" in renamed.columns else np.nan

    valid_gender = metadata["gender"].isin(["Female", "Male"])
    features = features.loc[valid_gender, feature_names].reset_index(drop=True)
    metadata = metadata.loc[valid_gender].reset_index(drop=True)

    assert features.shape[1] == 30
    assert list(features.columns) == feature_names
    return features, metadata


def load_reference_csv(path: str | Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    raw_df = pd.read_csv(path)
    print(f"CSV carregado: {path}")
    print(f"Linhas brutas: {len(raw_df)} | colunas brutas: {len(raw_df.columns)}")

    ignored = [col for col in AGGREGATED_EYE_COLUMNS if col in raw_df.columns]
    if ignored:
        print(f"Colunas agregadas de olhos ignoradas na matriz final: {ignored}")

    reference_features, reference_metadata = build_reference_feature_matrix(raw_df)

    print(f"Referências válidas: {len(reference_features)}")
    print(f"Amostras Female: {(reference_metadata['gender'] == 'Female').sum()}")
    print(f"Amostras Male: {(reference_metadata['gender'] == 'Male').sum()}")
    print("Colunas utilizadas:")
    print(reference_features.columns.tolist())

    return reference_features, reference_metadata


reference_features, reference_metadata = load_reference_csv(REFERENCE_CSV)

assert reference_features.shape[1] == 30
assert list(reference_features.columns) == feature_names
assert (reference_metadata["gender"] == "Female").any(), "Grupo Female vazio no CSV de referência."
assert (reference_metadata["gender"] == "Male").any(), "Grupo Male vazio no CSV de referência."


CSV carregado: imdb_shpm_reference_valid.csv
Linhas brutas: 1634 | colunas brutas: 33
Referências válidas: 1634
Amostras Female: 869
Amostras Male: 765
Colunas utilizadas:
['face_vs_nose', 'face_vs_mouth', 'face_vs_l_eye', 'face_vs_r_eye', 'face_vs_hair', 'nose_vs_face', 'nose_vs_mouth', 'nose_vs_l_eye', 'nose_vs_r_eye', 'nose_vs_hair', 'mouth_vs_face', 'mouth_vs_nose', 'mouth_vs_l_eye', 'mouth_vs_r_eye', 'mouth_vs_hair', 'l_eye_vs_face', 'l_eye_vs_nose', 'l_eye_vs_mouth', 'l_eye_vs_r_eye', 'l_eye_vs_hair', 'r_eye_vs_face', 'r_eye_vs_nose', 'r_eye_vs_mouth', 'r_eye_vs_l_eye', 'r_eye_vs_hair', 'hair_vs_face', 'hair_vs_nose', 'hair_vs_mouth', 'hair_vs_l_eye', 'hair_vs_r_eye']


## 8. Normalização Min-Max 

In [18]:
def clean_reference_features(features: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series, dict[str, int]]:
    clean = features.replace([np.inf, -np.inf], np.nan).copy()
    report = {
        "missing_before_imputation": int(clean.isna().sum().sum()),
        "infinite_before_imputation": int(np.isinf(features.to_numpy(dtype=float)).sum()),
    }
    impute_values = clean.median(axis=0, skipna=True).fillna(0.0)
    clean = clean.fillna(impute_values).fillna(0.0).astype(float)

    assert list(clean.columns) == feature_names
    assert np.isfinite(clean.to_numpy(dtype=float)).all()
    return clean, impute_values, report


def clean_query_features(features: pd.DataFrame, impute_values: pd.Series) -> pd.DataFrame:
    clean = features.loc[:, feature_names].replace([np.inf, -np.inf], np.nan).copy()
    clean = clean.fillna(impute_values).fillna(0.0).astype(float)

    assert list(clean.columns) == feature_names
    assert np.isfinite(clean.to_numpy(dtype=float)).all()
    return clean


reference_features_clean, reference_impute_values, missing_report = clean_reference_features(reference_features)

scaler = MinMaxScaler()
scaler.fit(reference_features_clean)

reference_normalized = scaler.transform(reference_features_clean)
reference_normalized_df = pd.DataFrame(reference_normalized, columns=feature_names)

female_reference_matrix = reference_normalized_df.loc[reference_metadata["gender"] == "Female"].to_numpy(dtype=float)
male_reference_matrix = reference_normalized_df.loc[reference_metadata["gender"] == "Male"].to_numpy(dtype=float)

assert female_reference_matrix.shape[0] > 0
assert male_reference_matrix.shape[0] > 0
assert female_reference_matrix.shape[1] == 30
assert male_reference_matrix.shape[1] == 30
assert np.isfinite(female_reference_matrix).all()
assert np.isfinite(male_reference_matrix).all()

print("Tratamento dos atributos de referência:")
print("1. Infinitos -> NaN.")
print("2. NaN -> mediana do atributo calculada apenas no IMDB.")
print("3. MinMaxScaler ajustado apenas no IMDB.")
print(missing_report)


Tratamento dos atributos de referência:
1. Infinitos -> NaN.
2. NaN -> mediana do atributo calculada apenas no IMDB.
3. MinMaxScaler ajustado apenas no IMDB.
{'missing_before_imputation': 0, 'infinite_before_imputation': 0}


## 9. Comparação 

In [19]:
def cosine_similarity(u, v):
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)

    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)

    if norm_u == 0 or norm_v == 0:
        return 0.0

    return float(np.dot(u, v) / (norm_u * norm_v))


def sigmoid_adjustment(score, mu=0.88, beta=10.0):
    return 1.0 / (1.0 + np.exp(-beta * (score - mu)))


def higher_group_from_delta(delta: float) -> str:
    if delta > 0:
        return "Female"
    if delta < 0:
        return "Male"
    return "Tie"


def comparison_status_from_delta_interval(delta_ci_lower: float, delta_ci_upper: float) -> str:
    if delta_ci_lower > 0:
        return "Female higher"
    if delta_ci_upper < 0:
        return "Male higher"
    return "Inconclusive"


def cosine_similarities_to_group(query_vector: np.ndarray, reference_matrix: np.ndarray) -> np.ndarray:
    similarities = np.asarray(
        [
            cosine_similarity(query_vector, reference_vector)
            for reference_vector in reference_matrix
        ],
        dtype=float,
    )
    similarities = similarities[np.isfinite(similarities)]
    if similarities.size == 0:
        raise ValueError("Nenhuma similaridade de cosseno válida foi encontrada para o grupo.")
    return similarities


def bootstrap_original_shpm_intervals(
    female_similarities: np.ndarray,
    male_similarities: np.ndarray,
    n_resamples: int = BOOTSTRAP_RESAMPLES,
    seed: int = RANDOM_SEED,
) -> dict[str, float | str]:
    if not COMPUTE_BOOTSTRAP:
        return {
            "female_ci_lower": np.nan,
            "female_ci_upper": np.nan,
            "male_ci_lower": np.nan,
            "male_ci_upper": np.nan,
            "delta_ci_lower": np.nan,
            "delta_ci_upper": np.nan,
            "comparison_status": "Inconclusive",
        }

    rng = np.random.default_rng(seed)
    female_similarities = np.asarray(female_similarities, dtype=float)
    male_similarities = np.asarray(male_similarities, dtype=float)
    female_similarities = female_similarities[np.isfinite(female_similarities)]
    male_similarities = male_similarities[np.isfinite(male_similarities)]

    if female_similarities.size == 0 or male_similarities.size == 0:
        raise ValueError("Bootstrap requer similaridades válidas para Female e Male.")

    boot_female = np.empty(n_resamples, dtype=float)
    boot_male = np.empty(n_resamples, dtype=float)
    boot_delta = np.empty(n_resamples, dtype=float)

    for index in range(n_resamples):
        female_sample = rng.choice(female_similarities, size=female_similarities.size, replace=True)
        male_sample = rng.choice(male_similarities, size=male_similarities.size, replace=True)

        female_score = float(sigmoid_adjustment(np.mean(female_sample), mu=SIGMOID_MU, beta=SIGMOID_BETA))
        male_score = float(sigmoid_adjustment(np.mean(male_sample), mu=SIGMOID_MU, beta=SIGMOID_BETA))

        boot_female[index] = female_score
        boot_male[index] = male_score
        boot_delta[index] = female_score - male_score

    female_ci_lower, female_ci_upper = np.percentile(boot_female, [2.5, 97.5])
    male_ci_lower, male_ci_upper = np.percentile(boot_male, [2.5, 97.5])
    delta_ci_lower, delta_ci_upper = np.percentile(boot_delta, [2.5, 97.5])

    return {
        "female_ci_lower": float(female_ci_lower),
        "female_ci_upper": float(female_ci_upper),
        "male_ci_lower": float(male_ci_lower),
        "male_ci_upper": float(male_ci_upper),
        "delta_ci_lower": float(delta_ci_lower),
        "delta_ci_upper": float(delta_ci_upper),
        "comparison_status": comparison_status_from_delta_interval(float(delta_ci_lower), float(delta_ci_upper)),
    }


def compare_query_with_reference(
    query_features: pd.DataFrame,
    seed: int,
) -> tuple[dict[str, float | str], dict[str, np.ndarray]]:
    assert list(query_features.columns) == feature_names

    query_clean = clean_query_features(query_features, reference_impute_values)
    query_vector = scaler.transform(query_clean)[0]
    query_vector = np.asarray(query_vector, dtype=float)
    assert np.isfinite(query_vector).all()

    female_similarities = cosine_similarities_to_group(query_vector, female_reference_matrix)
    male_similarities = cosine_similarities_to_group(query_vector, male_reference_matrix)

    mean_similarity_female = float(np.mean(female_similarities))
    mean_similarity_male = float(np.mean(male_similarities))

    shpm_female = float(sigmoid_adjustment(mean_similarity_female, mu=SIGMOID_MU, beta=SIGMOID_BETA))
    shpm_male = float(sigmoid_adjustment(mean_similarity_male, mu=SIGMOID_MU, beta=SIGMOID_BETA))
    delta_shpm = shpm_female - shpm_male

    intervals = bootstrap_original_shpm_intervals(
        female_similarities=female_similarities,
        male_similarities=male_similarities,
        n_resamples=BOOTSTRAP_RESAMPLES,
        seed=seed,
    )

    result = {
        "mean_cosine_female": mean_similarity_female,
        "mean_cosine_male": mean_similarity_male,
        "shpm_female": shpm_female,
        "shpm_male": shpm_male,
        "delta_shpm": delta_shpm,
        "absolute_difference": abs(delta_shpm),
        "higher_similarity_group": higher_group_from_delta(delta_shpm),
        **intervals,
    }

    assert np.isfinite([shpm_female, shpm_male, delta_shpm]).all()
    assert 0 <= shpm_female <= 1
    assert 0 <= shpm_male <= 1

    similarity_record = {
        "female_similarities": female_similarities,
        "male_similarities": male_similarities,
    }
    return result, similarity_record


## 10. Testes metodológicos

Estes testes verificam a forma dos atributos, a similaridade de cosseno, a sigmoide original, o intervalo dos scores, a estabilidade do bootstrap com a mesma semente e a separação dos grupos de referência.


In [20]:
def run_original_shpm_tests() -> None:
    assert len(feature_names) == 30
    assert len(set(feature_names)) == 30
    assert list(reference_features_clean.columns) == feature_names

    test_vector = np.arange(1, 31, dtype=float)
    assert np.isclose(cosine_similarity(test_vector, test_vector), 1.0)
    assert np.isclose(sigmoid_adjustment(0.88), 0.5)

    test_query_vector = reference_normalized_df.iloc[0].to_numpy(dtype=float)
    test_female_similarities = cosine_similarities_to_group(test_query_vector, female_reference_matrix)
    test_male_similarities = cosine_similarities_to_group(test_query_vector, male_reference_matrix)
    shpm_female = float(sigmoid_adjustment(np.mean(test_female_similarities), mu=SIGMOID_MU, beta=SIGMOID_BETA))
    shpm_male = float(sigmoid_adjustment(np.mean(test_male_similarities), mu=SIGMOID_MU, beta=SIGMOID_BETA))
    assert 0 <= shpm_female <= 1
    assert 0 <= shpm_male <= 1

    bootstrap_a = bootstrap_original_shpm_intervals(
        test_female_similarities,
        test_male_similarities,
        n_resamples=25,
        seed=RANDOM_SEED,
    )
    bootstrap_b = bootstrap_original_shpm_intervals(
        test_female_similarities,
        test_male_similarities,
        n_resamples=25,
        seed=RANDOM_SEED,
    )
    for key in [
        "female_ci_lower",
        "female_ci_upper",
        "male_ci_lower",
        "male_ci_upper",
        "delta_ci_lower",
        "delta_ci_upper",
    ]:
        assert np.isclose(bootstrap_a[key], bootstrap_b[key])

    female_mask = reference_metadata["gender"] == "Female"
    male_mask = reference_metadata["gender"] == "Male"
    assert female_mask.any()
    assert male_mask.any()
    assert not (female_mask & male_mask).any()

    outside_query = pd.DataFrame(
        [reference_features_clean.max(axis=0).to_numpy(dtype=float) * 3.0 + 1.0],
        columns=feature_names,
    )
    outside_vector = scaler.transform(clean_query_features(outside_query, reference_impute_values))[0]
    assert np.isfinite(outside_vector).all()


run_original_shpm_tests()
print("Testes do SHPM original concluídos com sucesso.")


Testes do SHPM original concluídos com sucesso.


## 11. Processamento em lote das imagens


In [21]:
VALID_IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}


def list_image_paths(folder: str | Path) -> list[Path]:
    folder = Path(folder)
    if not folder.exists():
        print(f"Pasta de imagens não encontrada: {folder}")
        return []

    image_paths = [
        path
        for path in folder.iterdir()
        if path.is_file() and path.suffix.casefold() in VALID_IMAGE_EXTENSIONS
    ]
    image_paths = sorted(image_paths, key=lambda path: path.name.casefold())
    assert image_paths == sorted(image_paths, key=lambda path: path.name.casefold())
    return image_paths


def empty_result_row(filename: str, status: str = "invalid", error_message: str = "") -> dict[str, Any]:
    row = {column: np.nan for column in RESULT_COLUMNS}
    row.update({"filename": filename, "status": status, "error_message": error_message})
    return row


def invalid_from_error(filename: str, error_message: str) -> tuple[dict[str, Any], dict[str, Any], None]:
    return (
        empty_result_row(filename=filename, status="invalid", error_message=error_message),
        make_empty_feature_row(filename=filename, status="invalid", error_message=error_message),
        None,
    )


def process_single_image(
    image_path: Path,
    sam3_runtime: tuple[Any, Any, str],
) -> tuple[dict[str, Any], dict[str, Any], dict[str, Any] | None]:
    result_row = empty_result_row(filename=image_path.name, status="invalid")
    feature_row = make_empty_feature_row(filename=image_path.name, status="invalid")
    plot_record = None

    try:
        _, processor, _ = sam3_runtime
        img_rgb = Image.open(image_path).convert("RGB")

        raw_masks, scores, part_errors = sam3_segment_parts(
            img_rgb=img_rgb,
            processor=processor,
            score_threshold=SCORE_THRESHOLD,
        )
        masks = clean_masks(raw_masks)
        areas = summarize_mask_areas(masks)

        save_segmentation_outputs(
            image_path=image_path,
            img_rgb=img_rgb,
            masks=masks,
            areas=areas,
            scores=scores,
        )

        validate_masks(areas=areas, part_errors=part_errors, min_face_area=MIN_FACE_AREA)

        shpm_values = extract_shpm_features_from_areas(areas)
        query_features = pd.DataFrame([shpm_values], columns=feature_names)
        comparison, similarity_record = compare_query_with_reference(
            query_features=query_features,
            seed=stable_seed(image_path.name),
        )

        result_row.update(
            {
                "status": "valid",
                "error_message": "",
                "face_area": areas["face"],
                "face_score": scores.get("face", np.nan),
                "nose_score": scores.get("nose", np.nan),
                "mouth_score": scores.get("mouth", np.nan),
                "l_eye_score": scores.get("l_eye", np.nan),
                "r_eye_score": scores.get("r_eye", np.nan),
                "hair_score": scores.get("hair", np.nan),
                **comparison,
            }
        )

        feature_row.update({"status": "valid", "error_message": "", **shpm_values})

        plot_record = {
            "filename": image_path.name,
            "image_path": image_path,
            "image_stem": safe_stem(image_path),
            **comparison,
            **similarity_record,
        }

    except Exception as exc:
        message = f"{type(exc).__name__}: {exc}"
        result_row.update({"status": "invalid", "error_message": message})
        feature_row.update({"status": "invalid", "error_message": message})

    return result_row, feature_row, plot_record


image_paths = list_image_paths(IMAGES_FOLDER)
print(f"Imagens encontradas: {len(image_paths)}")

results_rows: list[dict[str, Any]] = []
feature_rows: list[dict[str, Any]] = []
plot_records: list[dict[str, Any]] = []

sam3_runtime = None
sam3_initialization_error = None

if image_paths:
    try:
        sam3_runtime = initialize_sam3()
    except Exception as exc:
        sam3_initialization_error = f"{type(exc).__name__}: {exc}"
        print(f"Falha ao inicializar SAM 3. As imagens serão registradas como inválidas: {sam3_initialization_error}")

for image_path in tqdm(image_paths, desc="Processando imagens"):
    if sam3_initialization_error is not None:
        result_row, feature_row, plot_record = invalid_from_error(
            filename=image_path.name,
            error_message=f"SAM3 initialization failed: {sam3_initialization_error}",
        )
    else:
        result_row, feature_row, plot_record = process_single_image(image_path, sam3_runtime)

    results_rows.append(result_row)
    feature_rows.append(feature_row)
    if plot_record is not None:
        plot_records.append(plot_record)

results_df = pd.DataFrame(results_rows, columns=RESULT_COLUMNS)
features_df = pd.DataFrame(feature_rows)

for column in ["filename", "status", "error_message", *feature_names]:
    if column not in features_df.columns:
        features_df[column] = np.nan
features_df = features_df[["filename", "status", "error_message", *feature_names]]

valid_results_df = results_df.loc[results_df["status"] == "valid"].copy()
invalid_results_df = results_df.loc[results_df["status"] != "valid"].copy()

for _, row in valid_results_df.iterrows():
    scores = np.asarray([row["shpm_female"], row["shpm_male"]], dtype=float)
    assert np.isfinite(scores).all()
    assert ((scores >= 0.0) & (scores <= 1.0)).all()

print(f"Imagens válidas: {len(valid_results_df)}")
print(f"Imagens inválidas: {len(invalid_results_df)}")
if not invalid_results_df.empty:
    display(invalid_results_df[["filename", "error_message"]])


Imagens encontradas: 4
Dispositivo: cuda
Arquivo BPE: /workspace/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz
Model builder: /workspace/sam3/sam3/model_builder.py


config.json:   0%|          | 0.00/25.8k [00:00<?, ?B/s]

sam3.pt:   0%|          | 0.00/3.45G [00:00<?, ?B/s]

SAM 3 inicializado corretamente.


Processando imagens: 100%|██████████| 4/4 [00:30<00:00,  7.74s/it]

Imagens válidas: 4
Imagens inválidas: 0


## 12. Visualizações individuais

Cada imagem recebe um gráfico com SHPM Female e SHPM Male. As barras de erro representam o intervalo bootstrap quando disponível.


In [22]:
def make_errorbar_array(record: dict[str, Any]) -> np.ndarray | None:
    values = np.asarray([record["shpm_female"], record["shpm_male"]], dtype=float)
    lowers = np.asarray([record["female_ci_lower"], record["male_ci_lower"]], dtype=float)
    uppers = np.asarray([record["female_ci_upper"], record["male_ci_upper"]], dtype=float)
    all_values = np.concatenate([values, lowers, uppers])

    if not np.isfinite(all_values).all():
        return None

    lower_errors = np.maximum(values - lowers, 0.0)
    upper_errors = np.maximum(uppers - values, 0.0)
    return np.vstack([lower_errors, upper_errors])


def save_individual_shpm_plot(record: dict[str, Any], include_errorbars: bool = True) -> Path:
    img = Image.open(record["image_path"]).convert("RGB")
    width, height = img.size
    aspect = width / height
    fig_size = (8.0, 8.0 / aspect) if aspect >= 1 else (8.0 * aspect, 8.0)

    fig, ax_img = plt.subplots(figsize=fig_size, dpi=300)

    try:
        ax_img.imshow(img)
        ax_img.axis("off")
        fig.subplots_adjust(left=0, right=1, top=1, bottom=0)

        bar_ax = ax_img.inset_axes([0.23, 0.06, 0.54, 0.34])
        bar_ax.set_facecolor((1, 1, 1, 0.80))

        labels = ["SHPM Female", "SHPM Male"]
        x_positions = np.arange(len(labels), dtype=float)
        values = np.asarray([record["shpm_female"], record["shpm_male"]], dtype=float)
        colors = [BAR_COLORS["Women"], BAR_COLORS["Men"]]
        yerr = make_errorbar_array(record) if include_errorbars else None

        bars = bar_ax.bar(
            x_positions,
            values,
            width=0.65,
            color=colors,
            edgecolor="white",
            linewidth=1.2,
            yerr=yerr,
            capsize=3 if yerr is not None else 0,
        )

        bar_ax.set_xticks(x_positions)
        bar_ax.set_xticklabels(labels, fontsize=7)
        bar_ax.set_xlim(-0.6, len(labels) - 0.4)
        bar_ax.set_ylim(0, 1)
        bar_ax.set_ylabel("Original SHPM score", fontsize=7)
        bar_ax.tick_params(axis="y", labelsize=8)
        bar_ax.grid(axis="y", alpha=0.25)
        bar_ax.set_title(
            f"{record['filename']} | higher SHPM similarity: {record['higher_similarity_group']} | "
            f"difference: {record['delta_shpm']:+.3f} | {record['comparison_status']}",
            fontsize=7,
            pad=4,
        )

        for bar, value in zip(bars, values):
            text_y = max(min(value / 2.0, 0.95), 0.05)
            bar_ax.text(
                bar.get_x() + bar.get_width() / 2,
                text_y,
                f"{value:.3f}",
                ha="center",
                va="center",
                color="white",
                fontweight="bold",
                fontsize=8,
            )

        output_path = DIR_INDIVIDUAL_PLOTS / f"{record['image_stem']}_shpm_original.png"
        fig.savefig(output_path, dpi=300, bbox_inches="tight", pad_inches=0)
        return output_path

    finally:
        plt.close(fig)


individual_plot_paths = []
for record in plot_records:
    individual_plot_paths.append(save_individual_shpm_plot(record, include_errorbars=True))

print(f"Figuras individuais salvas: {len(individual_plot_paths)}")


Figuras individuais salvas: 4


## 13. Visualizações gerais

Os gráficos gerais resumem os scores por imagem, a diferença SHPM Female - SHPM Male, o ranking pela diferença absoluta, as distribuições de similaridade de cosseno e os intervalos bootstrap da diferença.


In [23]:
def save_summary_visualizations(valid_df: pd.DataFrame, records: list[dict[str, Any]]) -> pd.DataFrame:
    output_columns = [
        "filename",
        "mean_cosine_female",
        "mean_cosine_male",
        "shpm_female",
        "shpm_male",
        "delta_shpm",
        "absolute_difference",
        "higher_similarity_group",
        "comparison_status",
    ]

    if valid_df.empty:
        print("Nenhuma imagem válida para visualizações gerais.")
        return pd.DataFrame(columns=output_columns)

    DIR_SUMMARY.mkdir(parents=True, exist_ok=True)
    DIR_DIAGNOSTICS.mkdir(parents=True, exist_ok=True)

    plot_df = valid_df.copy().reset_index(drop=True)
    numeric_columns = ["shpm_female", "shpm_male", "delta_shpm", "absolute_difference"]
    for column in numeric_columns:
        plot_df[column] = pd.to_numeric(plot_df[column], errors="coerce")
    plot_df = plot_df.replace([np.inf, -np.inf], np.nan).dropna(subset=numeric_columns).reset_index(drop=True)

    if plot_df.empty:
        print("Nenhuma imagem possui valores SHPM válidos para os gráficos.")
        return pd.DataFrame(columns=output_columns)

    filenames = plot_df["filename"].astype(str).tolist()
    x_positions = np.arange(len(plot_df), dtype=float)
    fig_width = max(8.0, len(plot_df) * 1.15)

    fig, ax = plt.subplots(figsize=(fig_width, 5.5), dpi=180)
    try:
        bar_width = 0.38
        ax.bar(x_positions - bar_width / 2, plot_df["shpm_female"], width=bar_width, label="Female", color=BAR_COLORS["Women"])
        ax.bar(x_positions + bar_width / 2, plot_df["shpm_male"], width=bar_width, label="Male", color=BAR_COLORS["Men"])
        ax.set_ylim(0, 1)
        ax.set_ylabel("Original SHPM score")
        ax.set_title("Original SHPM: Female e Male")
        ax.set_xticks(x_positions)
        ax.set_xticklabels(filenames, rotation=45, ha="right")
        ax.legend()
        ax.grid(axis="y", alpha=0.25)
        fig.tight_layout()
        fig.savefig(DIR_SUMMARY / "grouped_bar_shpm_original.png", dpi=300, bbox_inches="tight")
    finally:
        plt.close(fig)

    fig, ax = plt.subplots(figsize=(fig_width, 5.0), dpi=180)
    try:
        colors = [BAR_COLORS["Women"] if value > 0 else BAR_COLORS["Men"] if value < 0 else "gray" for value in plot_df["delta_shpm"]]
        ax.bar(x_positions, plot_df["delta_shpm"], color=colors)
        ax.axhline(0, color="black", linewidth=1)
        ax.set_ylabel("SHPM Female - SHPM Male")
        ax.set_title("Diferença SHPM por imagem")
        ax.set_xticks(x_positions)
        ax.set_xticklabels(filenames, rotation=45, ha="right")
        ax.grid(axis="y", alpha=0.25)
        fig.tight_layout()
        fig.savefig(DIR_SUMMARY / "delta_shpm_original.png", dpi=300, bbox_inches="tight")
    finally:
        plt.close(fig)

    heatmap_df = plot_df.set_index("filename")[["shpm_female", "shpm_male"]].rename(columns={"shpm_female": "Female", "shpm_male": "Male"})
    fig_height = max(4.0, len(plot_df) * 0.55)
    fig, ax = plt.subplots(figsize=(6.5, fig_height), dpi=180)
    try:
        if sns is not None:
            sns.heatmap(heatmap_df, annot=True, fmt=".3f", cmap="mako", vmin=0, vmax=1, linewidths=0.5, linecolor="white", cbar_kws={"label": "Original SHPM score"}, ax=ax)
        else:
            image = ax.imshow(heatmap_df.to_numpy(dtype=float), cmap="viridis", vmin=0, vmax=1, aspect="auto")
            ax.set_xticks(np.arange(heatmap_df.shape[1]))
            ax.set_xticklabels(heatmap_df.columns)
            ax.set_yticks(np.arange(heatmap_df.shape[0]))
            ax.set_yticklabels(heatmap_df.index)
            fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
        ax.set_title("Heatmap SHPM original")
        ax.set_xlabel("Grupo de referência")
        ax.set_ylabel("Imagem")
        fig.tight_layout()
        fig.savefig(DIR_SUMMARY / "heatmap_shpm_original.png", dpi=300, bbox_inches="tight")
    finally:
        plt.close(fig)

    ranked = plot_df.sort_values("absolute_difference", ascending=False)[output_columns].reset_index(drop=True)
    ranked.to_csv(DIR_SUMMARY / "shpm_difference_ranked_table.csv", index=False)

    record_by_name = {record["filename"]: record for record in records}
    fig, axes = plt.subplots(len(plot_df), 1, figsize=(8.0, max(3.0, len(plot_df) * 2.2)), dpi=170, squeeze=False)
    try:
        for ax, (_, row) in zip(axes.ravel(), plot_df.iterrows()):
            record = record_by_name.get(row["filename"])
            if record is None:
                ax.axis("off")
                continue
            bins = np.linspace(-1, 1, 41)
            ax.hist(record["female_similarities"], bins=bins, alpha=0.55, color=BAR_COLORS["Women"], label="Female")
            ax.hist(record["male_similarities"], bins=bins, alpha=0.55, color=BAR_COLORS["Men"], label="Male")
            ax.set_title(row["filename"])
            ax.set_xlabel("Cosine similarity")
            ax.set_ylabel("Count")
            ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(DIR_DIAGNOSTICS / "cosine_similarity_distributions.png", dpi=300, bbox_inches="tight")
    finally:
        plt.close(fig)

    fig, ax = plt.subplots(figsize=(fig_width, 5.0), dpi=180)
    try:
        yerr = np.vstack([
            np.maximum(plot_df["delta_shpm"].to_numpy(dtype=float) - plot_df["delta_ci_lower"].to_numpy(dtype=float), 0.0),
            np.maximum(plot_df["delta_ci_upper"].to_numpy(dtype=float) - plot_df["delta_shpm"].to_numpy(dtype=float), 0.0),
        ])
        ax.errorbar(x_positions, plot_df["delta_shpm"], yerr=yerr, fmt="o", color="#4c78a8", capsize=4)
        ax.axhline(0, color="black", linewidth=1)
        ax.set_ylabel("SHPM Female - SHPM Male")
        ax.set_title("Intervalos bootstrap da diferença SHPM")
        ax.set_xticks(x_positions)
        ax.set_xticklabels(filenames, rotation=45, ha="right")
        ax.grid(axis="y", alpha=0.25)
        fig.tight_layout()
        fig.savefig(DIR_SUMMARY / "bootstrap_delta_intervals.png", dpi=300, bbox_inches="tight")
    finally:
        plt.close(fig)

    print(f"Visualizações gerais salvas em: {DIR_SUMMARY}")
    return ranked


## 14. Exportação dos resultados


In [24]:
results_path = OUTPUT_ROOT / "shpm_original_results.csv"
features_path = OUTPUT_ROOT / "extracted_shpm_features.csv"

ranked_summary_df = save_summary_visualizations(valid_results_df, plot_records)

results_df.to_csv(results_path, index=False)
features_df.to_csv(features_path, index=False)

processed_count = int((results_df["status"] == "valid").sum()) if not results_df.empty else 0
invalid_count = int((results_df["status"] != "valid").sum()) if not results_df.empty else 0

print(f"Número de imagens encontradas: {len(image_paths)}")
print(f"Número de imagens processadas: {processed_count}")
print(f"Número de imagens inválidas: {invalid_count}")
print(f"Caminho dos resultados: {OUTPUT_ROOT.resolve()}")
print(f"CSV final: {results_path}")
print(f"Atributos SHPM extraídos: {features_path}")

summary_columns = [
    "filename",
    "mean_cosine_female",
    "mean_cosine_male",
    "shpm_female",
    "shpm_male",
    "delta_shpm",
    "absolute_difference",
    "higher_similarity_group",
    "comparison_status",
]

if results_df.empty:
    display(pd.DataFrame(columns=summary_columns))
else:
    display(results_df.loc[results_df["status"] == "valid", summary_columns])


Visualizações gerais salvas em: outputs_shpm_original/summary
Número de imagens encontradas: 4
Número de imagens processadas: 4
Número de imagens inválidas: 0
Caminho dos resultados: /workspace/sam3/sam3/outputs_shpm_original
CSV final: outputs_shpm_original/shpm_original_results.csv
Atributos SHPM extraídos: outputs_shpm_original/extracted_shpm_features.csv


,filename,mean_cosine_female,mean_cosine_male,shpm_female,shpm_male,delta_shpm,absolute_difference,higher_similarity_group,comparison_status
0,Flux_SHPM.png,0.870068,0.800840,0.475191,0.311824,0.163366,0.163366,Female,Female higher
1,Gemini.png,0.835061,0.812668,0.389505,0.337753,0.051752,0.051752,Female,Female higher
2,gpt.png,0.722303,0.806332,0.171225,0.323731,-0.152505,0.152505,Male,Male higher
3,SDXL.png,0.752040,0.812379,0.217619,0.337109,-0.119490,0.119490,Male,Male higher
